# EcoSort AI — TrashNet Training with MobileNetV2

**B.Tech Deep Learning project notebook**

This notebook trains and evaluates a six-class waste image classifier for:

`cardboard · glass · metal · paper · plastic · trash`

It uses ImageNet-pretrained **MobileNetV2**, a frozen-backbone transfer-learning phase, and a guarded fine-tuning phase. The workflow downloads and verifies the official TrashNet resized archive, audits image quality and duplicates, makes leakage-resistant train/validation/test splits, evaluates exactly once after model selection, and exports deployable artifacts.

> **Before running:** in Google Colab choose **Runtime → Change runtime type → GPU**, then choose **Runtime → Run all**. No Kaggle token, Google Drive mount, or manual dataset upload is required.


## Method and reproducibility notes

- **Source:** the original author's pinned [TrashNet Hugging Face archive](https://huggingface.co/datasets/garythung/trashnet), with the pinned [original GitHub archive](https://github.com/garythung/trashnet) as fallback. Each download has its own expected SHA-256 digest.
- **Data integrity:** ZIP checksum/CRC, folder layout, official class counts, Pillow decoding, dimensions, exact pixel hashes, and perceptual hashes are checked.
- **Leakage control:** contradictory exact duplicates are quarantined; same-label exact duplicates are deduplicated; detected near-duplicate clusters stay in one split. Because TrashNet does not provide object/session identifiers, visually different photographs of the same physical object cannot be ruled out with absolute certainty.
- **Honest evaluation:** model and fine-tuning choices use validation data only. The test split remains untouched until the final candidate is selected.
- **Dataset limitation:** TrashNet is small and was photographed mainly against a light background. Strong test performance here does not by itself prove robustness to real sorting-line conditions.

Original project citation: Gary Thung and Mindy Yang, *Classification of Trash for Recyclability Status*, Stanford CS229 (2016). [Project report](https://cs229.stanford.edu/proj2016/report/ThungYang-ClassificationOfTrashForRecyclabilityStatus-report.pdf)


## 1. Install lightweight dependencies

Colab normally supplies a CUDA-compatible TensorFlow build. This cell installs packages only when they are missing or below the notebook's minimum supported version, avoiding an unnecessary TensorFlow replacement on a normal Colab GPU runtime.


In [ ]:
import importlib.util
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version

if importlib.util.find_spec("packaging") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "packaging>=23"])
from packaging.requirements import Requirement

required_packages = {
    "tensorflow": "tensorflow>=2.15",
    "numpy": "numpy>=1.23",
    "pandas": "pandas>=1.5",
    "PIL": "Pillow>=9.5",
    "sklearn": "scikit-learn>=1.3",
    "matplotlib": "matplotlib>=3.7",
    "seaborn": "seaborn>=0.12",
    "requests": "requests>=2.31",
    "imagehash": "ImageHash>=4.3.1,<5",
}

packages_to_install = []
for module, requirement_text in required_packages.items():
    requirement = Requirement(requirement_text)
    try:
        installed_version = version(requirement.name)
        version_is_supported = installed_version in requirement.specifier
    except PackageNotFoundError:
        installed_version = None
        version_is_supported = False
    module_is_available = importlib.util.find_spec(module) is not None
    if not module_is_available or not version_is_supported:
        packages_to_install.append(requirement_text)

if packages_to_install:
    print("Installing/upgrading required packages:", packages_to_install)
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "--quiet", *packages_to_install]
    )
else:
    print("All required packages already meet the supported versions.")


## 2. Imports, deterministic seeds, and runtime check

A fixed seed makes the split and initialization reproducible. GPU kernels can still vary slightly across hardware or TensorFlow versions, so the notebook records the runtime configuration alongside the metrics.


In [ ]:
import gc
import hashlib
import json
import math
import os
import platform
import random
import shutil
import time
import warnings
import zipfile
from datetime import datetime, timezone
from pathlib import Path

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import imagehash
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import seaborn as sns
import tensorflow as tf
from IPython.display import FileLink, display
from PIL import Image, ImageOps
from requests.adapters import HTTPAdapter
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.utils.class_weight import compute_class_weight
from urllib3.util.retry import Retry

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
try:
    tf.config.experimental.enable_op_determinism()
    determinism_status = "enabled"
except Exception as exc:
    determinism_status = f"not enabled ({exc})"

pd.set_option("display.max_colwidth", 120)
sns.set_theme(style="whitegrid", context="notebook")

gpu_devices = tf.config.list_physical_devices("GPU")
print(f"Python: {platform.python_version()}")
print(f"TensorFlow: {tf.__version__}")
print(f"Deterministic TensorFlow ops: {determinism_status}")
print(f"GPU devices: {gpu_devices or 'none detected'}")
if not gpu_devices:
    warnings.warn(
        "No GPU was detected. The notebook will still run, but training will be much slower. "
        "In Colab, select Runtime → Change runtime type → GPU and run again."
    )


## 3. Configuration

The 224×224 input size is MobileNetV2's standard ImageNet configuration. The approximate 71/14/14 split comes from grouped stratified folds, which keeps every detected duplicate cluster together.


In [ ]:
CLASS_NAMES = ["cardboard", "glass", "metal", "paper", "plastic", "trash"]
CLASS_TO_INDEX = {name: index for index, name in enumerate(CLASS_NAMES)}
EXPECTED_CLASS_COUNTS = {
    "cardboard": 403,
    "glass": 501,
    "metal": 410,
    "paper": 594,
    "plastic": 482,
    "trash": 137,
}
EXPECTED_TOTAL_IMAGES = sum(EXPECTED_CLASS_COUNTS.values())

IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
INITIAL_EPOCHS = 25
FINE_TUNE_EPOCHS = 12
INITIAL_LEARNING_RATE = 1e-3
FINE_TUNE_LEARNING_RATE = 1e-5
DROPOUT_RATE = 0.35
FINE_TUNE_LAYERS = 40
EARLY_STOPPING_PATIENCE = 5

ENABLE_FINE_TUNING = True
MIN_VAL_ACCURACY_FOR_FINE_TUNING = 0.55
MAX_TRAIN_VAL_GAP_FOR_FINE_TUNING = 0.20
PERCEPTUAL_HASH_DISTANCE = 6

# Seven outer folds and six inner folds produce approximately 71/14/14.
OUTER_SPLITS = 7
INNER_SPLITS = 6

WORK_DIR = Path("/content/ecosort_ai")
DATA_DIR = WORK_DIR / "data"
EXTRACT_DIR = DATA_DIR / "extracted"
CHECKPOINT_DIR = WORK_DIR / "checkpoints"
ARTIFACT_DIR = WORK_DIR / "artifacts"
ARCHIVE_PATH = DATA_DIR / "dataset-resized.zip"

for directory in (DATA_DIR, CHECKPOINT_DIR, ARTIFACT_DIR):
    directory.mkdir(parents=True, exist_ok=True)

DOWNLOAD_SOURCES = [
    {
        "name": "Original-author Hugging Face mirror (pinned revision)",
        "url": (
            "https://huggingface.co/datasets/garythung/trashnet/resolve/"
            "94cd17ebfd6702bf62281d3c89f22ff649815b46/"
            "dataset-resized.zip?download=true"
        ),
        "sha256": "c060e8abfe5d6de0578ca15be1ed8ad0794a865d333c3473d53d1d9ad6e38b8c",
        "bytes": 42_834_870,
    },
    {
        "name": "Original GitHub repository (pinned commit fallback)",
        "url": (
            "https://raw.githubusercontent.com/garythung/trashnet/"
            "c536e5eeb7399d1a8912118e7447f8493c9d3413/data/dataset-resized.zip"
        ),
        "sha256": "0bf472790f8b20e5c950d5b5012a9d38af0d3392efd65f8ce171334fc16b07c2",
        "bytes": 42_834_870,
    },
]

print({
    "classes": CLASS_NAMES,
    "image_size": IMAGE_SIZE,
    "batch_size": BATCH_SIZE,
    "initial_epochs_max": INITIAL_EPOCHS,
    "fine_tune_epochs_max": FINE_TUNE_EPOCHS,
    "workspace": str(WORK_DIR),
})


## 4. Download and cryptographically verify TrashNet

The primary and fallback ZIP files contain the same image payloads but different ZIP metadata, so each has a distinct trusted digest. Downloads are streamed to a temporary file, checked for exact byte size and SHA-256, then atomically promoted.


In [ ]:
def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def identify_known_archive(path):
    if not path.exists():
        return None
    observed_size = path.stat().st_size
    observed_sha = sha256_file(path)
    for source in DOWNLOAD_SOURCES:
        if observed_size == source["bytes"] and observed_sha == source["sha256"]:
            return source
    return None


cached_source = identify_known_archive(ARCHIVE_PATH)
if cached_source:
    active_archive_source = cached_source
    print(f"Using verified cached archive: {cached_source['name']}")
else:
    if ARCHIVE_PATH.exists():
        print("Removing an unrecognized or incomplete cached archive.")
        ARCHIVE_PATH.unlink()

    retry_policy = Retry(
        total=2,
        connect=2,
        read=2,
        backoff_factor=1,
        status_forcelist=(429, 500, 502, 503, 504),
        allowed_methods=frozenset(["GET"]),
    )
    session = requests.Session()
    session.headers.update({"User-Agent": "EcoSort-AI-Colab/1.0"})
    session.mount("https://", HTTPAdapter(max_retries=retry_policy))

    active_archive_source = None
    download_errors = []
    for source_index, source in enumerate(DOWNLOAD_SOURCES):
        partial_path = ARCHIVE_PATH.with_suffix(f".source{source_index}.part")
        if partial_path.exists():
            partial_path.unlink()
        try:
            print(f"Downloading: {source['name']}")
            with session.get(source["url"], stream=True, timeout=(20, 180)) as response:
                response.raise_for_status()
                with open(partial_path, "wb") as handle:
                    for chunk in response.iter_content(chunk_size=1024 * 1024):
                        if chunk:
                            handle.write(chunk)

            observed_size = partial_path.stat().st_size
            observed_sha = sha256_file(partial_path)
            if observed_size != source["bytes"]:
                raise ValueError(
                    f"Size mismatch: expected {source['bytes']:,}, got {observed_size:,} bytes"
                )
            if observed_sha != source["sha256"]:
                raise ValueError(
                    f"SHA-256 mismatch: expected {source['sha256']}, got {observed_sha}"
                )
            partial_path.replace(ARCHIVE_PATH)
            active_archive_source = source
            print("Download verified successfully.")
            break
        except Exception as exc:
            download_errors.append(f"{source['name']}: {type(exc).__name__}: {exc}")
            print(f"Source failed verification: {exc}")
        finally:
            if partial_path.exists():
                partial_path.unlink()

    if active_archive_source is None:
        raise RuntimeError(
            "All official TrashNet sources failed. Details:\n- " + "\n- ".join(download_errors)
        )

ARCHIVE_SHA256 = sha256_file(ARCHIVE_PATH)
assert zipfile.is_zipfile(ARCHIVE_PATH), "The verified download is not a ZIP archive."
with zipfile.ZipFile(ARCHIVE_PATH) as archive:
    bad_member = archive.testzip()
    assert bad_member is None, f"ZIP CRC validation failed at member: {bad_member}"

print(f"Archive: {ARCHIVE_PATH}")
print(f"Size: {ARCHIVE_PATH.stat().st_size / (1024 ** 2):.2f} MiB")
print(f"SHA-256: {ARCHIVE_SHA256}")
print(f"Verified source: {active_archive_source['name']}")


## 5. Safely extract and locate the six class folders

The extractor rejects path traversal, encrypted members, and symbolic-link members. It ignores macOS metadata by scanning only direct image files inside the six expected folders.


In [ ]:
def safe_extract_zip(archive_path, destination):
    destination = destination.resolve()
    with zipfile.ZipFile(archive_path) as archive:
        for info in archive.infolist():
            member_path = (destination / info.filename).resolve()
            if os.path.commonpath([str(destination), str(member_path)]) != str(destination):
                raise ValueError(f"Unsafe ZIP member path: {info.filename}")
            if info.flag_bits & 0x1:
                raise ValueError(f"Encrypted ZIP member is not allowed: {info.filename}")
            unix_file_type = (info.external_attr >> 16) & 0o170000
            if unix_file_type == 0o120000:
                raise ValueError(f"Symbolic-link ZIP member is not allowed: {info.filename}")
        archive.extractall(destination)


# Re-extract on every run so a manually modified prior extraction can never be trusted
# merely because a cache marker still exists. The archive is small enough for this to be cheap.
if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
safe_extract_zip(ARCHIVE_PATH, EXTRACT_DIR)
print("Archive freshly extracted and validated.")


def locate_class_root(extraction_root, expected_classes):
    candidates = [extraction_root]
    candidates.extend(path for path in extraction_root.rglob("*") if path.is_dir())
    candidates.sort(key=lambda path: len(path.parts))
    expected = set(expected_classes)
    for candidate in candidates:
        direct_directories = {path.name for path in candidate.iterdir() if path.is_dir()}
        if expected.issubset(direct_directories):
            return candidate
    raise FileNotFoundError(
        f"Could not find a directory containing all expected classes: {expected_classes}"
    )


DATASET_ROOT = locate_class_root(EXTRACT_DIR, CLASS_NAMES)
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
class_paths = {
    class_name: sorted(
        path for path in (DATASET_ROOT / class_name).iterdir()
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
    )
    for class_name in CLASS_NAMES
}
raw_counts = {class_name: len(paths) for class_name, paths in class_paths.items()}

assert raw_counts == EXPECTED_CLASS_COUNTS, (
    "The archive folder counts do not match the canonical TrashNet release. "
    f"Expected {EXPECTED_CLASS_COUNTS}, observed {raw_counts}."
)
assert sum(raw_counts.values()) == EXPECTED_TOTAL_IMAGES

print(f"Dataset root: {DATASET_ROOT}")
display(pd.DataFrame({
    "class_name": CLASS_NAMES,
    "image_count": [raw_counts[name] for name in CLASS_NAMES],
    "percentage": [100 * raw_counts[name] / EXPECTED_TOTAL_IMAGES for name in CLASS_NAMES],
}).style.format({"percentage": "{:.2f}%"}))


## 6. Decode every image and build an audit manifest

Pillow first verifies each file, then reopens and fully decodes it. The manifest records file and normalized-pixel SHA-256 hashes plus two 64-bit perceptual hashes. Corrupt images would be reported and excluded; the pinned canonical archive is expected to contain none.


In [ ]:
def inspect_image(path, class_name):
    file_hash = sha256_file(path)
    with Image.open(path) as probe:
        original_mode = probe.mode
        probe.verify()
    with Image.open(path) as source_image:
        image = ImageOps.exif_transpose(source_image).convert("RGB")
        image.load()
        width, height = image.size
        pixel_digest = hashlib.sha256()
        pixel_digest.update(f"RGB:{width}x{height}:".encode("utf-8"))
        pixel_digest.update(np.asarray(image, dtype=np.uint8).tobytes())
        return {
            "path": str(path),
            "relative_path": str(path.relative_to(DATASET_ROOT)),
            "class_name": class_name,
            "file_bytes": path.stat().st_size,
            "width": width,
            "height": height,
            "original_mode": original_mode,
            "file_sha256": file_hash,
            "pixel_sha256": pixel_digest.hexdigest(),
            "phash": str(imagehash.phash(image, hash_size=8)),
            "dhash": str(imagehash.dhash(image, hash_size=8)),
        }


audit_rows = []
corrupt_rows = []
all_labeled_paths = [
    (path, class_name)
    for class_name in CLASS_NAMES
    for path in class_paths[class_name]
]
for number, (path, class_name) in enumerate(all_labeled_paths, start=1):
    try:
        audit_rows.append(inspect_image(path, class_name))
    except Exception as exc:
        corrupt_rows.append({
            "path": str(path),
            "class_name": class_name,
            "error": f"{type(exc).__name__}: {exc}",
        })
    if number % 500 == 0 or number == len(all_labeled_paths):
        print(f"Audited {number:,}/{len(all_labeled_paths):,} files")

manifest_raw = pd.DataFrame(audit_rows).sort_values(
    ["class_name", "relative_path"]
).reset_index(drop=True)
corrupt_df = pd.DataFrame(corrupt_rows, columns=["path", "class_name", "error"])

print(f"Valid images: {len(manifest_raw):,}")
print(f"Corrupt/unreadable images: {len(corrupt_df):,}")
if not corrupt_df.empty:
    display(corrupt_df)
    warnings.warn("Corrupt images were excluded from the manifest.")

assert len(manifest_raw) + len(corrupt_df) == EXPECTED_TOTAL_IMAGES
assert corrupt_df.empty, "The pinned canonical archive should contain no corrupt class images."
assert len(manifest_raw) == EXPECTED_TOTAL_IMAGES
assert not manifest_raw.empty
assert set(manifest_raw["class_name"]) == set(CLASS_NAMES)

dimension_summary = (
    manifest_raw.groupby(["width", "height"])
    .size()
    .rename("count")
    .reset_index()
    .sort_values("count", ascending=False)
)
display(dimension_summary)
observed_dimensions = set(zip(manifest_raw["width"], manifest_raw["height"]))
assert observed_dimensions == {(512, 384)}, (
    f"Unexpected image dimensions in the canonical resized dataset: {observed_dimensions}"
)


### Class distribution and representative samples

TrashNet is imbalanced: the `trash` class is much smaller than `paper`. Balanced training-only class weights are therefore used later; validation and test metrics remain unweighted.


In [ ]:
audited_counts = manifest_raw["class_name"].value_counts().reindex(CLASS_NAMES)
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
sns.barplot(x=audited_counts.index, y=audited_counts.values, ax=axes[0], color="#2a9d8f")
axes[0].set(title="TrashNet class counts", xlabel="Class", ylabel="Images")
axes[0].tick_params(axis="x", rotation=30)
for patch, value in zip(axes[0].patches, audited_counts.values):
    axes[0].annotate(str(value), (patch.get_x() + patch.get_width() / 2, value),
                     ha="center", va="bottom")

axes[1].axis("off")
axes[1].text(
    0.0, 0.95,
    "Imbalance summary\n\n" + "\n".join(
        f"{name:10s}: {audited_counts[name]:3d} ({100 * audited_counts[name] / len(manifest_raw):5.1f}%)"
        for name in CLASS_NAMES
    ),
    va="top", family="monospace", fontsize=12,
)
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, len(CLASS_NAMES), figsize=(18, 3.2))
for axis, class_name in zip(axes, CLASS_NAMES):
    sample_path = Path(manifest_raw.loc[manifest_raw["class_name"] == class_name, "path"].iloc[0])
    with Image.open(sample_path) as image:
        axis.imshow(ImageOps.exif_transpose(image).convert("RGB"))
    axis.set_title(class_name)
    axis.axis("off")
plt.suptitle("One verified sample from each class", y=1.02)
plt.tight_layout()
plt.show()


## 7. Remove exact conflicts and group perceptual near-duplicates

The canonical archive contains three known pairs with identical image content but contradictory labels. Training on one copy and testing on the other would be both leakage and an impossible labeling target, so **all members of every cross-label exact-duplicate group are quarantined**. Same-label exact repeats, if present, are reduced to one canonical row.

For the remaining images, a conservative near-duplicate link requires *both* pHash and dHash Hamming distance ≤ the configured threshold. Connected images form one group that cannot cross split boundaries. Perceptual matches are not deleted because they may be legitimate distinct samples.


In [ ]:
pixel_group_label_counts = manifest_raw.groupby("pixel_sha256")["class_name"].nunique()
conflicting_hashes = set(pixel_group_label_counts[pixel_group_label_counts > 1].index)
conflict_df = manifest_raw[manifest_raw["pixel_sha256"].isin(conflicting_hashes)].copy()

if not conflict_df.empty:
    print(f"Quarantining {len(conflict_df)} files from {len(conflicting_hashes)} cross-label exact groups:")
    display(conflict_df[["relative_path", "class_name", "file_sha256", "pixel_sha256"]])
else:
    print("No cross-label exact duplicates were found.")

manifest_without_conflicts = manifest_raw[
    ~manifest_raw["pixel_sha256"].isin(conflicting_hashes)
].copy()
same_label_duplicate_mask = manifest_without_conflicts.duplicated(
    subset=["class_name", "pixel_sha256"], keep="first"
)
same_label_duplicates = manifest_without_conflicts[same_label_duplicate_mask].copy()
if not same_label_duplicates.empty:
    print(f"Removing {len(same_label_duplicates)} redundant same-label exact copies.")
    display(same_label_duplicates[["relative_path", "class_name", "pixel_sha256"]])

manifest = manifest_without_conflicts[~same_label_duplicate_mask].copy()
manifest = manifest.sort_values(["class_name", "relative_path"]).reset_index(drop=True)

assert not manifest["pixel_sha256"].duplicated().any()
assert set(manifest["class_name"]) == set(CLASS_NAMES)
print(f"Usable unique, non-conflicting images: {len(manifest):,}")
display(manifest["class_name"].value_counts().reindex(CLASS_NAMES).rename("usable_count").to_frame())


In [ ]:
class UnionFind:
    def __init__(self, size):
        self.parent = list(range(size))
        self.rank = [0] * size

    def find(self, item):
        while self.parent[item] != item:
            self.parent[item] = self.parent[self.parent[item]]
            item = self.parent[item]
        return item

    def union(self, left, right):
        left_root, right_root = self.find(left), self.find(right)
        if left_root == right_root:
            return
        if self.rank[left_root] < self.rank[right_root]:
            left_root, right_root = right_root, left_root
        self.parent[right_root] = left_root
        if self.rank[left_root] == self.rank[right_root]:
            self.rank[left_root] += 1


phashes = [int(value, 16) for value in manifest["phash"]]
dhashes = [int(value, 16) for value in manifest["dhash"]]
union_find = UnionFind(len(manifest))
near_pair_count = 0
near_pair_preview = []

for left in range(len(manifest)):
    for right in range(left):
        phash_distance = (phashes[left] ^ phashes[right]).bit_count()
        if phash_distance > PERCEPTUAL_HASH_DISTANCE:
            continue
        dhash_distance = (dhashes[left] ^ dhashes[right]).bit_count()
        if dhash_distance <= PERCEPTUAL_HASH_DISTANCE:
            union_find.union(left, right)
            near_pair_count += 1
            if len(near_pair_preview) < 20:
                near_pair_preview.append({
                    "image_a": manifest.at[right, "relative_path"],
                    "image_b": manifest.at[left, "relative_path"],
                    "pHash_distance": phash_distance,
                    "dHash_distance": dhash_distance,
                })

roots = [union_find.find(index) for index in range(len(manifest))]
root_to_group = {root: f"duplicate_group_{number:04d}"
                 for number, root in enumerate(sorted(set(roots)))}
manifest["duplicate_group"] = [root_to_group[root] for root in roots]

group_sizes = manifest.groupby("duplicate_group").size()
multi_image_groups = group_sizes[group_sizes > 1]
cross_label_near_groups = (
    manifest.groupby("duplicate_group")["class_name"].nunique() > 1
).sum()

print(f"Perceptual near-duplicate links detected: {near_pair_count:,}")
print(f"Multi-image connected groups: {len(multi_image_groups):,}")
print(f"Largest connected group: {int(group_sizes.max())}")
print(f"Connected groups spanning labels: {int(cross_label_near_groups)}")
if near_pair_preview:
    display(pd.DataFrame(near_pair_preview))

DATA_QUALITY_SUMMARY = {
    "raw_images": int(len(manifest_raw)),
    "corrupt_images_excluded": int(len(corrupt_df)),
    "cross_label_exact_images_quarantined": int(len(conflict_df)),
    "cross_label_exact_groups": int(len(conflicting_hashes)),
    "same_label_exact_copies_removed": int(len(same_label_duplicates)),
    "usable_images": int(len(manifest)),
    "perceptual_near_duplicate_links": int(near_pair_count),
    "multi_image_duplicate_groups": int(len(multi_image_groups)),
    "cross_label_perceptual_groups": int(cross_label_near_groups),
}


## 8. Create leakage-resistant train, validation, and test splits

`StratifiedGroupKFold` balances labels while treating each detected duplicate cluster as indivisible. Among candidate folds, the notebook chooses the holdout closest to the full class distribution. The test fold is selected first; validation is then selected only from the remaining data.


In [ ]:
manifest["label_id"] = manifest["class_name"].map(CLASS_TO_INDEX).astype("int32")


def best_grouped_holdout(frame, n_splits, seed):
    splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    all_counts = np.bincount(frame["label_id"], minlength=len(CLASS_NAMES)).astype(float)
    target_counts = all_counts / n_splits
    target_size = len(frame) / n_splits
    best = None

    for fold_number, (keep_positions, holdout_positions) in enumerate(
        splitter.split(
            X=np.zeros(len(frame)),
            y=frame["label_id"].to_numpy(),
            groups=frame["duplicate_group"].to_numpy(),
        )
    ):
        holdout_labels = frame.iloc[holdout_positions]["label_id"].to_numpy()
        holdout_counts = np.bincount(holdout_labels, minlength=len(CLASS_NAMES)).astype(float)
        if np.any(holdout_counts == 0):
            continue
        distribution_error = np.mean(
            np.abs(holdout_counts - target_counts) / np.maximum(target_counts, 1.0)
        )
        size_error = abs(len(holdout_positions) - target_size) / target_size
        score = distribution_error + 0.25 * size_error
        candidate = (score, fold_number, keep_positions, holdout_positions)
        if best is None or candidate[0] < best[0]:
            best = candidate

    if best is None:
        raise RuntimeError("Could not create a grouped holdout containing every class.")
    return best[2], best[3], best[1], best[0]


trainval_positions, test_positions, test_fold, test_score = best_grouped_holdout(
    manifest, OUTER_SPLITS, SEED
)
trainval_df = manifest.iloc[trainval_positions].reset_index(drop=True)
test_df = manifest.iloc[test_positions].reset_index(drop=True)

train_positions, val_positions, val_fold, val_score = best_grouped_holdout(
    trainval_df, INNER_SPLITS, SEED + 1
)
train_df = trainval_df.iloc[train_positions].reset_index(drop=True)
val_df = trainval_df.iloc[val_positions].reset_index(drop=True)

split_frames = {"train": train_df, "validation": val_df, "test": test_df}
for split_name, frame in split_frames.items():
    assert set(frame["class_name"]) == set(CLASS_NAMES), f"{split_name} is missing a class"


def assert_pairwise_disjoint(field):
    split_sets = {name: set(frame[field]) for name, frame in split_frames.items()}
    names = list(split_sets)
    for left_index, left_name in enumerate(names):
        for right_name in names[left_index + 1:]:
            overlap = split_sets[left_name] & split_sets[right_name]
            assert not overlap, (
                f"Leakage detected in {field} between {left_name} and {right_name}: "
                f"{list(overlap)[:5]}"
            )


for leakage_key in ("path", "file_sha256", "pixel_sha256", "duplicate_group"):
    assert_pairwise_disjoint(leakage_key)

split_table = pd.DataFrame({
    split_name: frame["class_name"].value_counts().reindex(CLASS_NAMES, fill_value=0)
    for split_name, frame in split_frames.items()
})
split_table.loc["TOTAL"] = split_table.sum(axis=0)
split_table["all_splits"] = split_table.sum(axis=1)
display(split_table)

print(f"Selected outer test fold: {test_fold} (balance score {test_score:.4f})")
print(f"Selected inner validation fold: {val_fold} (balance score {val_score:.4f})")
for split_name, frame in split_frames.items():
    print(f"{split_name:10s}: {len(frame):4d} images ({100 * len(frame) / len(manifest):5.2f}%)")
print("Leakage assertions passed for paths, exact hashes, and perceptual groups.")

SPLIT_COUNTS = {name: int(len(frame)) for name, frame in split_frames.items()}


## 9. Build efficient `tf.data` pipelines

Images remain in the `[0, 255]` range after resizing. MobileNetV2 preprocessing is applied inside the model exactly once. Only the training dataset is shuffled, and random augmentation layers activate only while `training=True`; validation, test, and exported-model inference are never augmented.


In [ ]:
AUTOTUNE = tf.data.AUTOTUNE


def decode_and_resize(path, label):
    image_bytes = tf.io.read_file(path)
    image = tf.io.decode_image(image_bytes, channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, IMAGE_SIZE, method="bilinear", antialias=True)
    image = tf.cast(image, tf.float32)
    return image, tf.cast(label, tf.int32)


def make_dataset(frame, training=False):
    paths = frame["path"].astype(str).to_numpy()
    labels = frame["label_id"].astype("int32").to_numpy()
    dataset = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        dataset = dataset.shuffle(
            buffer_size=len(frame), seed=SEED, reshuffle_each_iteration=True
        )
    dataset = dataset.map(decode_and_resize, num_parallel_calls=AUTOTUNE)
    dataset = dataset.batch(BATCH_SIZE, drop_remainder=False)
    dataset = dataset.prefetch(AUTOTUNE)
    options = tf.data.Options()
    options.experimental_deterministic = True
    return dataset.with_options(options)


train_ds = make_dataset(train_df, training=True)
val_ds = make_dataset(val_df, training=False)
test_ds = make_dataset(test_df, training=False)

class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(len(CLASS_NAMES)),
    y=train_df["label_id"].to_numpy(),
)
class_weights = {index: float(weight) for index, weight in enumerate(class_weights_array)}
print("Training-only class weights:")
display(pd.DataFrame({
    "class_name": CLASS_NAMES,
    "label_id": range(len(CLASS_NAMES)),
    "class_weight": [class_weights[index] for index in range(len(CLASS_NAMES))],
}).style.format({"class_weight": "{:.3f}"}))

batch_images, batch_labels = next(iter(train_ds))
print("Batch image shape:", batch_images.shape)
print("Batch label shape:", batch_labels.shape)
print("Raw resized pixel range:", float(tf.reduce_min(batch_images)), "to", float(tf.reduce_max(batch_images)))


## 10. Build the MobileNetV2 transfer-learning model

The initial phase freezes the complete ImageNet backbone. Global average pooling keeps the head compact, and dropout reduces overfitting. `Rescaling(1/127.5, offset=-1)` is exactly MobileNetV2's required mapping from `[0,255]` pixels to `[-1,1]`.


In [ ]:
tf.keras.backend.clear_session()

data_augmentation = tf.keras.Sequential(
    [
        tf.keras.layers.RandomFlip("horizontal", seed=SEED),
        tf.keras.layers.RandomRotation(0.05, fill_mode="reflect", seed=SEED + 1),
        tf.keras.layers.RandomTranslation(0.05, 0.05, fill_mode="reflect", seed=SEED + 2),
        tf.keras.layers.RandomZoom(0.10, 0.10, fill_mode="reflect", seed=SEED + 3),
        tf.keras.layers.RandomContrast(0.10, seed=SEED + 4),
    ],
    name="training_augmentation",
)

inputs = tf.keras.Input(shape=(*IMAGE_SIZE, 3), name="image")
x = data_augmentation(inputs)
x = tf.keras.layers.Rescaling(
    scale=1.0 / 127.5, offset=-1.0, name="mobilenetv2_preprocessing"
)(x)
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(*IMAGE_SIZE, 3),
    include_top=False,
    weights="imagenet",
)
base_model.trainable = False
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D(name="global_average_pooling")(x)
x = tf.keras.layers.Dropout(DROPOUT_RATE, seed=SEED, name="dropout")(x)
outputs = tf.keras.layers.Dense(
    len(CLASS_NAMES), activation="softmax", dtype="float32", name="class_probabilities"
)(x)
model = tf.keras.Model(inputs, outputs, name="ecosort_mobilenetv2")

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=INITIAL_LEARNING_RATE),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=[tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy")],
)

assert not base_model.trainable
assert model.output_shape == (None, len(CLASS_NAMES))
model.summary()


### Visual check of training-only augmentation

The preview deliberately calls the augmentation layer with `training=True`. During validation, testing, and normal prediction Keras calls it with `training=False`, so images pass through unchanged.


In [ ]:
preview_images = batch_images[:6]
augmented_preview = data_augmentation(preview_images, training=True)
fig, axes = plt.subplots(2, 6, figsize=(16, 6))
for index in range(6):
    axes[0, index].imshow(tf.cast(tf.clip_by_value(preview_images[index], 0, 255), tf.uint8))
    axes[0, index].set_title(f"Original\n{CLASS_NAMES[int(batch_labels[index])]}")
    axes[1, index].imshow(tf.cast(tf.clip_by_value(augmented_preview[index], 0, 255), tf.uint8))
    axes[1, index].set_title("Augmented")
    axes[0, index].axis("off")
    axes[1, index].axis("off")
plt.suptitle("Augmentation is used only during training")
plt.tight_layout()
plt.show()


## 11. Train the classification head

- **EarlyStopping** halts when validation loss stops improving and restores the best weights.
- **ReduceLROnPlateau** lowers the learning rate after a shorter plateau.
- **ModelCheckpoint** preserves the best complete `.keras` candidate independently of in-memory state.


In [ ]:
INITIAL_CHECKPOINT = CHECKPOINT_DIR / "initial_best.keras"
FINE_TUNE_CHECKPOINT = CHECKPOINT_DIR / "fine_tuned_best.keras"


def make_callbacks(checkpoint_path, phase_name):
    return [
        tf.keras.callbacks.ModelCheckpoint(
            filepath=str(checkpoint_path),
            monitor="val_loss",
            mode="min",
            save_best_only=True,
            verbose=1,
        ),
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            mode="min",
            patience=EARLY_STOPPING_PATIENCE,
            restore_best_weights=True,
            verbose=1,
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            mode="min",
            factor=0.3,
            patience=2,
            min_lr=1e-7,
            verbose=1,
        ),
        tf.keras.callbacks.TerminateOnNaN(),
        tf.keras.callbacks.CSVLogger(
            str(CHECKPOINT_DIR / f"{phase_name}_history.csv"), append=False
        ),
    ]


if INITIAL_CHECKPOINT.exists():
    INITIAL_CHECKPOINT.unlink()

history_initial = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=INITIAL_EPOCHS,
    class_weight=class_weights,
    callbacks=make_callbacks(INITIAL_CHECKPOINT, "initial"),
    verbose=1,
)
assert INITIAL_CHECKPOINT.exists(), "The initial best-model checkpoint was not created."
print(f"Initial phase completed after {len(history_initial.epoch)} epoch(s).")


## 12. Guarded fine-tuning of upper MobileNetV2 layers

Fine-tuning is attempted only when the best initial validation accuracy clears a minimum and the corresponding train–validation gap is not severe. The notebook reloads the best initial checkpoint, unfreezes only the top backbone layers, keeps Batch Normalization layers frozen, **recompiles** with a much smaller learning rate, and writes a separate checkpoint. Later, the initial and fine-tuned candidates compete using validation loss only.


In [ ]:
initial_best_epoch = int(np.argmin(history_initial.history["val_loss"]))
initial_best_val_loss = float(history_initial.history["val_loss"][initial_best_epoch])
initial_best_val_accuracy = float(history_initial.history["val_accuracy"][initial_best_epoch])
initial_train_accuracy_at_best = float(history_initial.history["accuracy"][initial_best_epoch])
initial_generalization_gap = initial_train_accuracy_at_best - initial_best_val_accuracy

should_fine_tune = (
    ENABLE_FINE_TUNING
    and initial_best_val_accuracy >= MIN_VAL_ACCURACY_FOR_FINE_TUNING
    and initial_generalization_gap <= MAX_TRAIN_VAL_GAP_FOR_FINE_TUNING
)

print({
    "best_initial_epoch_1_based": initial_best_epoch + 1,
    "validation_loss": initial_best_val_loss,
    "validation_accuracy": initial_best_val_accuracy,
    "training_accuracy": initial_train_accuracy_at_best,
    "train_minus_validation_accuracy": initial_generalization_gap,
    "fine_tuning_approved": should_fine_tune,
})

history_fine = None
if should_fine_tune:
    model = tf.keras.models.load_model(INITIAL_CHECKPOINT)
    backbone_candidates = [
        layer for layer in model.layers
        if isinstance(layer, tf.keras.Model) and "mobilenetv2" in layer.name.lower()
    ]
    if len(backbone_candidates) != 1:
        raise RuntimeError(
            f"Expected one MobileNetV2 backbone after reload, found {len(backbone_candidates)}."
        )
    base_model = backbone_candidates[0]
    base_model.trainable = True
    fine_tune_start = max(0, len(base_model.layers) - FINE_TUNE_LAYERS)
    for layer in base_model.layers[:fine_tune_start]:
        layer.trainable = False
    for layer in base_model.layers[fine_tune_start:]:
        if isinstance(layer, tf.keras.layers.BatchNormalization):
            layer.trainable = False

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=FINE_TUNE_LEARNING_RATE),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(),
        metrics=[tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy")],
    )
    if FINE_TUNE_CHECKPOINT.exists():
        FINE_TUNE_CHECKPOINT.unlink()

    trainable_backbone_layers = sum(layer.trainable for layer in base_model.layers)
    print(
        f"Fine-tuning {trainable_backbone_layers} of {len(base_model.layers)} backbone layers "
        f"from a learning rate of {FINE_TUNE_LEARNING_RATE:g}."
    )
    history_fine = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=FINE_TUNE_EPOCHS,
        class_weight=class_weights,
        callbacks=make_callbacks(FINE_TUNE_CHECKPOINT, "fine_tune"),
        verbose=1,
    )
    assert FINE_TUNE_CHECKPOINT.exists(), "The fine-tuned checkpoint was not created."
else:
    reasons = []
    if not ENABLE_FINE_TUNING:
        reasons.append("fine-tuning was disabled")
    if initial_best_val_accuracy < MIN_VAL_ACCURACY_FOR_FINE_TUNING:
        reasons.append("initial validation accuracy was below the safety threshold")
    if initial_generalization_gap > MAX_TRAIN_VAL_GAP_FOR_FINE_TUNING:
        reasons.append("the initial train–validation gap indicated overfitting")
    print("Fine-tuning skipped: " + "; ".join(reasons))


## 13. Plot learning curves and select the final candidate

The phase boundary is shown when fine-tuning ran. Candidate selection uses validation loss, with validation accuracy only as a tie-breaker. The test set is still untouched at this point.


In [ ]:
def history_frame(history, phase, epoch_offset=0):
    frame = pd.DataFrame(history.history)
    frame["epoch"] = np.arange(1, len(frame) + 1) + epoch_offset
    frame["phase"] = phase
    return frame


history_parts = [history_frame(history_initial, "Frozen backbone", 0)]
if history_fine is not None:
    history_parts.append(
        history_frame(history_fine, "Fine-tuning", len(history_initial.epoch))
    )
combined_history = pd.concat(history_parts, ignore_index=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(combined_history["epoch"], combined_history["accuracy"], label="Train")
axes[0].plot(combined_history["epoch"], combined_history["val_accuracy"], label="Validation")
axes[0].set(title="Accuracy by epoch", xlabel="Epoch", ylabel="Accuracy")
axes[0].legend()

axes[1].plot(combined_history["epoch"], combined_history["loss"], label="Train")
axes[1].plot(combined_history["epoch"], combined_history["val_loss"], label="Validation")
axes[1].set(title="Loss by epoch", xlabel="Epoch", ylabel="Cross-entropy loss")
axes[1].legend()

if history_fine is not None:
    boundary = len(history_initial.epoch) + 0.5
    for axis in axes:
        axis.axvline(boundary, linestyle="--", color="black", alpha=0.6, label="Fine-tune start")
plt.tight_layout()
plt.show()


In [ ]:
candidate_paths = {"initial": INITIAL_CHECKPOINT}
if history_fine is not None:
    candidate_paths["fine_tuned"] = FINE_TUNE_CHECKPOINT

validation_candidates = {}
for candidate_name, candidate_path in candidate_paths.items():
    candidate_model = tf.keras.models.load_model(candidate_path)
    candidate_result = candidate_model.evaluate(val_ds, return_dict=True, verbose=0)
    validation_candidates[candidate_name] = {
        key: float(value) for key, value in candidate_result.items()
    }
    del candidate_model
    gc.collect()

SELECTED_STAGE = min(
    validation_candidates,
    key=lambda name: (
        validation_candidates[name]["loss"],
        -validation_candidates[name]["accuracy"],
    ),
)
SELECTED_CHECKPOINT = candidate_paths[SELECTED_STAGE]

display(pd.DataFrame(validation_candidates).T.rename_axis("candidate"))
print(f"Selected candidate (validation only): {SELECTED_STAGE}")
print(f"Checkpoint: {SELECTED_CHECKPOINT}")


## 14. Final evaluation on the untouched test set

The chosen model is now evaluated on test data for the first time. Reported precision, recall, and F1 include both **macro averages** (each class counts equally) and **weighted averages** (classes contribute by support). No values are hard-coded.


In [ ]:
final_model = tf.keras.models.load_model(SELECTED_CHECKPOINT)
keras_test_result = final_model.evaluate(test_ds, return_dict=True, verbose=1)
test_probabilities = final_model.predict(test_ds, verbose=1)
y_test = test_df["label_id"].to_numpy(dtype=np.int32)
y_pred = np.argmax(test_probabilities, axis=1).astype(np.int32)
test_confidence = np.max(test_probabilities, axis=1)

assert len(y_test) == len(y_pred) == len(test_df)
assert test_probabilities.shape == (len(test_df), len(CLASS_NAMES))
assert np.allclose(test_probabilities.sum(axis=1), 1.0, atol=1e-4)

test_metrics = {
    "loss": float(keras_test_result["loss"]),
    "accuracy": float(accuracy_score(y_test, y_pred)),
    "precision_macro": float(precision_score(y_test, y_pred, average="macro", zero_division=0)),
    "recall_macro": float(recall_score(y_test, y_pred, average="macro", zero_division=0)),
    "f1_macro": float(f1_score(y_test, y_pred, average="macro", zero_division=0)),
    "precision_weighted": float(precision_score(y_test, y_pred, average="weighted", zero_division=0)),
    "recall_weighted": float(recall_score(y_test, y_pred, average="weighted", zero_division=0)),
    "f1_weighted": float(f1_score(y_test, y_pred, average="weighted", zero_division=0)),
}
assert math.isclose(test_metrics["accuracy"], float(keras_test_result["accuracy"]), abs_tol=1e-6)

display(pd.Series(test_metrics, name="value").to_frame().style.format("{:.4f}"))


### Confusion matrix

Rows are true labels and columns are predicted labels. The left plot shows counts; the right plot normalizes each true-class row to expose class-specific error patterns despite imbalance.


In [ ]:
confusion_counts = confusion_matrix(
    y_test, y_pred, labels=np.arange(len(CLASS_NAMES))
)
confusion_normalized = confusion_matrix(
    y_test, y_pred, labels=np.arange(len(CLASS_NAMES)), normalize="true"
)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.heatmap(
    confusion_counts, annot=True, fmt="d", cmap="Blues", cbar=False,
    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[0]
)
axes[0].set(title="Confusion matrix — counts", xlabel="Predicted", ylabel="True")
sns.heatmap(
    confusion_normalized, annot=True, fmt=".2f", cmap="Greens", vmin=0, vmax=1,
    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[1]
)
axes[1].set(title="Confusion matrix — row normalized", xlabel="Predicted", ylabel="True")
plt.tight_layout()
plt.show()


### Classification report

Support is the number of untouched test images in each class. A per-class report is especially important here because overall accuracy can hide weakness on the small `trash` class.


In [ ]:
classification_report_dict = classification_report(
    y_test,
    y_pred,
    labels=np.arange(len(CLASS_NAMES)),
    target_names=CLASS_NAMES,
    output_dict=True,
    zero_division=0,
)
classification_report_text = classification_report(
    y_test,
    y_pred,
    labels=np.arange(len(CLASS_NAMES)),
    target_names=CLASS_NAMES,
    digits=4,
    zero_division=0,
)
print(classification_report_text)


## 15. Real prediction examples

The first grid samples up to two test images from every class. Titles show the true label, predicted label, and softmax confidence. Green is correct; red is incorrect. The second grid focuses explicitly on the model's highest-confidence mistakes.


In [ ]:
prediction_frame = test_df[["path", "relative_path", "class_name", "label_id"]].copy()
prediction_frame["predicted_id"] = y_pred
prediction_frame["predicted_class"] = [CLASS_NAMES[index] for index in y_pred]
prediction_frame["confidence"] = test_confidence
prediction_frame["correct"] = prediction_frame["label_id"] == prediction_frame["predicted_id"]


def show_prediction_grid(frame, title, max_images=12, columns=4):
    frame = frame.head(max_images).reset_index(drop=True)
    if frame.empty:
        print(f"{title}: no images to display.")
        return
    rows = math.ceil(len(frame) / columns)
    fig, axes = plt.subplots(rows, columns, figsize=(4 * columns, 4 * rows))
    axes = np.atleast_1d(axes).ravel()
    for axis in axes:
        axis.axis("off")
    for axis, (_, row) in zip(axes, frame.iterrows()):
        with Image.open(row["path"]) as image:
            axis.imshow(ImageOps.exif_transpose(image).convert("RGB"))
        title_color = "darkgreen" if row["correct"] else "crimson"
        axis.set_title(
            f"True: {row['class_name']}\nPred: {row['predicted_class']} "
            f"({100 * row['confidence']:.1f}%)",
            color=title_color,
            fontsize=10,
        )
        axis.axis("off")
    fig.suptitle(title, fontsize=15)
    plt.tight_layout()
    plt.show()


balanced_examples = (
    prediction_frame.groupby("class_name", group_keys=False)
    .sample(n=2, random_state=SEED)
    .reset_index(drop=True)
)
show_prediction_grid(
    balanced_examples,
    "Test predictions — up to two examples per true class",
    max_images=12,
    columns=4,
)


In [ ]:
misclassified = prediction_frame[~prediction_frame["correct"]].sort_values(
    "confidence", ascending=False
)
print(f"Misclassified test images: {len(misclassified)}/{len(prediction_frame)}")
if misclassified.empty:
    print("No test misclassifications occurred in this run.")
else:
    show_prediction_grid(
        misclassified,
        "Highest-confidence misclassified test examples",
        max_images=12,
        columns=4,
    )
    display(
        misclassified[
            ["relative_path", "class_name", "predicted_class", "confidence"]
        ].head(20).style.format({"confidence": "{:.2%}"})
    )


## 16. Export the final model and metadata

Three standalone artifacts are produced:

1. `ecosort_mobilenetv2.keras` — selected end-to-end Keras model, including preprocessing.
2. `class_names.json` — the exact output-index mapping and input/preprocessing contract.
3. `model_metrics.json` — actual evaluation results, split sizes, data audit, configuration, confusion matrix, and classification report.


In [ ]:
FINAL_MODEL_PATH = ARTIFACT_DIR / "ecosort_mobilenetv2.keras"
CLASS_NAMES_PATH = ARTIFACT_DIR / "class_names.json"
MODEL_METRICS_PATH = ARTIFACT_DIR / "model_metrics.json"

final_model.save(FINAL_MODEL_PATH, overwrite=True)

class_mapping_payload = {
    "class_names": CLASS_NAMES,
    "class_to_index": CLASS_TO_INDEX,
    "index_to_class": {str(index): name for index, name in enumerate(CLASS_NAMES)},
    "input_shape": [IMAGE_SIZE[0], IMAGE_SIZE[1], 3],
    "input_dtype": "float32",
    "raw_input_range": [0, 255],
    "preprocessing": {
        "name": "MobileNetV2",
        "included_in_model": True,
        "operation": "x / 127.5 - 1.0",
        "model_range": [-1, 1],
    },
    "output": "softmax probabilities in class_names order",
}
with open(CLASS_NAMES_PATH, "w", encoding="utf-8") as handle:
    json.dump(class_mapping_payload, handle, indent=2, ensure_ascii=False, allow_nan=False)


def to_json_safe(value):
    if isinstance(value, dict):
        return {str(key): to_json_safe(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_json_safe(item) for item in value]
    if isinstance(value, np.ndarray):
        return to_json_safe(value.tolist())
    if isinstance(value, np.generic):
        return to_json_safe(value.item())
    if isinstance(value, float):
        if not math.isfinite(value):
            raise ValueError(f"Non-finite metric cannot be serialized: {value}")
        return value
    return value


metrics_payload = {
    "project": "EcoSort AI",
    "model": "MobileNetV2 transfer learning",
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "tensorflow_version": tf.__version__,
    "python_version": platform.python_version(),
    "gpu_devices": [device.name for device in gpu_devices],
    "random_seed": SEED,
    "dataset": {
        "name": "TrashNet resized",
        "verified_source": active_archive_source["name"],
        "source_url": active_archive_source["url"],
        "archive_sha256": ARCHIVE_SHA256,
        "canonical_raw_class_counts": EXPECTED_CLASS_COUNTS,
        "quality_audit": DATA_QUALITY_SUMMARY,
        "usable_class_counts": {
            name: int((manifest["class_name"] == name).sum()) for name in CLASS_NAMES
        },
    },
    "splits": {
        "method": "nested StratifiedGroupKFold with detected duplicate groups",
        "sizes": SPLIT_COUNTS,
        "class_counts": {
            split_name: {
                name: int((frame["class_name"] == name).sum()) for name in CLASS_NAMES
            }
            for split_name, frame in split_frames.items()
        },
    },
    "training": {
        "image_size": list(IMAGE_SIZE),
        "batch_size": BATCH_SIZE,
        "initial_epochs_completed": int(len(history_initial.epoch)),
        "fine_tune_epochs_completed": int(len(history_fine.epoch)) if history_fine is not None else 0,
        "initial_learning_rate": INITIAL_LEARNING_RATE,
        "fine_tune_learning_rate": FINE_TUNE_LEARNING_RATE,
        "dropout_rate": DROPOUT_RATE,
        "class_weights": {CLASS_NAMES[index]: weight for index, weight in class_weights.items()},
        "fine_tuning_attempted": history_fine is not None,
        "fine_tuning_decision": {
            "initial_best_val_accuracy": initial_best_val_accuracy,
            "initial_train_val_accuracy_gap": initial_generalization_gap,
            "minimum_val_accuracy_required": MIN_VAL_ACCURACY_FOR_FINE_TUNING,
            "maximum_gap_allowed": MAX_TRAIN_VAL_GAP_FOR_FINE_TUNING,
        },
    },
    "model_selection": {
        "selected_stage": SELECTED_STAGE,
        "criterion": "lowest validation loss; validation accuracy tie-breaker",
        "validation_candidates": validation_candidates,
        "test_data_used_for_selection": False,
    },
    "test_metrics": test_metrics,
    "confusion_matrix_labels": CLASS_NAMES,
    "confusion_matrix": confusion_counts.tolist(),
    "classification_report": classification_report_dict,
}
metrics_payload = to_json_safe(metrics_payload)
with open(MODEL_METRICS_PATH, "w", encoding="utf-8") as handle:
    json.dump(metrics_payload, handle, indent=2, ensure_ascii=False, allow_nan=False)

print("Exported artifacts:")
for artifact in (FINAL_MODEL_PATH, CLASS_NAMES_PATH, MODEL_METRICS_PATH):
    print(f"- {artifact.name}: {artifact.stat().st_size / (1024 ** 2):.2f} MiB")


## 17. Reload and validate all exported artifacts

A production artifact should not merely save—it should reload. This cell checks model deserialization without custom objects, output shape/probability validity, inference equivalence, JSON round trips, exact class order, and non-empty files.


In [ ]:
for artifact in (FINAL_MODEL_PATH, CLASS_NAMES_PATH, MODEL_METRICS_PATH):
    assert artifact.exists() and artifact.stat().st_size > 0, f"Missing/empty artifact: {artifact}"

reloaded_model = tf.keras.models.load_model(FINAL_MODEL_PATH)
sample_batch = next(iter(test_ds))[0][:2]
original_output = final_model.predict(sample_batch, verbose=0)
reloaded_output = reloaded_model.predict(sample_batch, verbose=0)

assert reloaded_model.output_shape == (None, len(CLASS_NAMES))
assert np.allclose(original_output, reloaded_output, rtol=1e-5, atol=1e-6)
assert np.allclose(reloaded_output.sum(axis=1), 1.0, atol=1e-4)

with open(CLASS_NAMES_PATH, encoding="utf-8") as handle:
    reloaded_mapping = json.load(handle)
with open(MODEL_METRICS_PATH, encoding="utf-8") as handle:
    reloaded_metrics = json.load(handle)

assert reloaded_mapping["class_names"] == CLASS_NAMES
assert reloaded_metrics["model_selection"]["selected_stage"] == SELECTED_STAGE
assert math.isclose(
    reloaded_metrics["test_metrics"]["accuracy"], test_metrics["accuracy"], abs_tol=1e-12
)

print("Artifact validation passed.")
for artifact in (FINAL_MODEL_PATH, CLASS_NAMES_PATH, MODEL_METRICS_PATH):
    print(f"{artifact.name}\n  SHA-256: {sha256_file(artifact)}")


## 18. Create downloadable copies

The three exact artifacts are copied to Colab's `/content` root and linked below. The final lines also invoke Colab's download API for each file; your browser may ask you to allow multiple downloads. The links and Files pane remain available as fallbacks.


In [ ]:
downloadable_paths = []
for source_path in (FINAL_MODEL_PATH, CLASS_NAMES_PATH, MODEL_METRICS_PATH):
    destination_path = Path("/content") / source_path.name
    shutil.copy2(source_path, destination_path)
    assert sha256_file(source_path) == sha256_file(destination_path)
    downloadable_paths.append(destination_path)

print("Download the three validated artifacts:")
for downloadable_path in downloadable_paths:
    display(FileLink(str(downloadable_path)))

try:
    from google.colab import files as colab_files

    print("Starting browser downloads (allow multiple downloads if prompted)...")
    for downloadable_path in downloadable_paths:
        colab_files.download(str(downloadable_path))
        time.sleep(1)
except ImportError:
    print("Not running in Colab; use the file links above.")
except Exception as exc:
    warnings.warn(f"Automatic browser download did not complete: {exc}. Use the links above.")

print("\nAll EcoSort AI training-stage work is complete. No website, API, or deployment was created.")
